# DART LLM Fine-tuning
EXAONE-3.5-2.4B-Instruct + QLoRA (4-bit)
RTX 4070 Ti (12GB VRAM) 기준

In [ ]:
import json
import torch
from pathlib import Path
from trl import SFTTrainer
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. 설정

In [ ]:
MODEL_ID   = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
DATASET    = '../data/processed/dart_qa_dataset.jsonl'
LORA_OUT   = '../outputs/lora'
MERGED_OUT = '../outputs/merged'


LORA_R       = 16
LORA_ALPHA   = 32
BATCH_SIZE   = 4
GRAD_ACCUM   = 4    
MAX_SEQ_LEN  = 2048
EPOCHS       = 3
LR           = 2e-4

## 2. 데이터셋 로드 및 포맷

In [ ]:
# JSONL 로드
records = []
with open(DATASET, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

print(f'총 {len(records)}건')

# EXAONE 채팅 포맷으로 변환
SYSTEM = '당신은 한국 금융 공시(DART) 전문 AI 어시스턴트입니다. 공시 본문을 바탕으로 핵심 내용을 정확하고 간결하게 답변합니다.'

def format_prompt(rec):
    user = f'[공시 본문]\n{rec["input"][:1500]}\n\n[질문]\n{rec["instruction"]}' if rec.get('input') else rec['instruction']
    return {
        'text': (
            f'[|system|]{SYSTEM}[|endofturn|]'
            f'[|user|]{user}[|endofturn|]'
            f'[|assistant|]{rec["output"]}[|endofturn|]'
        )
    }

data = Dataset.from_list([format_prompt(r) for r in records])
data = data.train_test_split(test_size=0.1, seed=42)
print(f'train: {len(data["train"])} / eval: {len(data["test"])}')

## 3. 모델 로드 (4-bit QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('모델 로드 완료')

## 4. LoRA 설정

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. 학습

In [ ]:
training_args = TrainingArguments(
    output_dir=LORA_OUT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data['train'],
    eval_dataset=data['test'],
    args=training_args,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
)

trainer.train()

## 6. LoRA 병합 → 전체 모델 저장

In [ ]:
from peft import PeftModel

# 4-bit 모델은 병합 전 fp16으로 재로드 필요
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_model, LORA_OUT)
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_OUT)
tokenizer.save_pretrained(MERGED_OUT)
print(f'병합 완료: {MERGED_OUT}')

## 7. 추론 테스트

In [ ]:
test_input = '이 사업보고서에서 회사의 주요 사업 내용을 요약해줘.'
test_context = records[0]['input'][:500]

messages = [
    {'role': 'system', 'content': SYSTEM},
    {'role': 'user', 'content': f'[공시 본문]\n{test_context}\n\n[질문]\n{test_input}'},
]

inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
with torch.no_grad():
    outputs = merged.generate(inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))